# Ollama Worker Demo

This notebook demonstrates the structured local worker flow:

1. Create a task packet
2. Create a worker
3. Run the worker with a live Ollama-backed task
4. Inspect worker state
5. Resume the last packet
6. Close the worker

The notebook captures the `worker_id` programmatically. No manual copy/paste is required.

In [1]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

def is_repo_root(candidate: Path) -> bool:
    return (candidate / "src" / "main.py").exists() and (candidate / "tests" / "test_porting_workspace.py").exists()

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if is_repo_root(candidate):
            return candidate
        for nested in (candidate / "claw-code", candidate / "harness" / "claw-code"):
            if is_repo_root(nested):
                return nested
    raise RuntimeError(
        f"Could not locate the claw-code repo from {current}. Start the notebook from the repo, its notebooks directory, or the enclosing autoresearch workspace."
    )

ROOT = find_repo_root()
PYTHON = sys.executable
NOTEBOOK_DIR = ROOT / "notebooks"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(args: list[str], check: bool = True, timeout: int = 120) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(shlex.quote(arg) for arg in args))
    result = subprocess.run(args, cwd=ROOT, capture_output=True, text=True, timeout=timeout)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        details = result.stderr.strip() or result.stdout.strip() or "no output"
        raise RuntimeError(f"command failed with exit code {result.returncode}: {details}")
    return result

def run_json(args: list[str], check: bool = True, timeout: int = 120):
    result = run_cmd(args, check=check, timeout=timeout)
    return json.loads(result.stdout)

def ollama_ready(host: str = "http://localhost:11434") -> bool:
    try:
        with urllib.request.urlopen(f"{host.rstrip('/')}/api/tags", timeout=2) as response:
            return response.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False

print(f"Workspace root: {ROOT}")
print(f"Python: {PYTHON}")

Workspace root: /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code
Python: /Users/wongdowling/opt/anaconda3/bin/python


In [2]:
OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5-coder:7b"
OLLAMA_IS_READY = ollama_ready(OLLAMA_HOST)
print({"ollama_host": OLLAMA_HOST, "ollama_ready": OLLAMA_IS_READY})
if not OLLAMA_IS_READY:
    raise RuntimeError("Ollama is not reachable. Start Ollama before running the live smoke test.")

{'ollama_host': 'http://localhost:11434', 'ollama_ready': True}


In [3]:
task_packet = {
    "objective": "Read src/main.py lines 1 to 40 and return the list of top-level subcommands registered by build_parser().",
    "scope": "Read src/main.py only. Do not edit any file. Do not run tests.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
        "python3 -c \"import src.main; print('import ok')\"",
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Return the subcommand names as a JSON array under the key 'subcommands'.",
    "escalation_policy": "Stop if Ollama is unavailable or a required tool fails repeatedly.",
}

task_path = NOTEBOOK_DIR / "demo_task.json"
task_path.write_text(json.dumps(task_packet, indent=2))
print("task packet written to:", task_path)
task_packet


task packet written to: /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/demo_task.json


{'objective': 'Read src/main.py lines 1 to 40 and return the list of top-level subcommands registered by build_parser().',
 'scope': 'Read src/main.py only. Do not edit any file. Do not run tests.',
 'repo': 'claw-code',
 'branch_policy': 'Do not create or switch branches.',
 'acceptance_tests': ['python3 -c "import src.main; print(\'import ok\')"'],
 'commit_policy': 'Do not commit.',
 'reporting_contract': "Return the subcommand names as a JSON array under the key 'subcommands'.",
 'escalation_policy': 'Stop if Ollama is unavailable or a required tool fails repeatedly.'}

In [4]:
worker = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "create",
    "--root",
    str(ROOT),
    "--model",
    OLLAMA_MODEL,
    "--host",
    OLLAMA_HOST,
])
worker_id = worker["worker_id"]
worker

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker create --root /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code --model qwen2.5-coder:7b --host http://localhost:11434
{
  "worker_id": "666c92adca9e42a6a6214639cc5a8aa0",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "ready",
  "created_at": "2026-04-08T01:53:21Z",
  "updated_at": "2026-04-08T01:53:21Z",
  "run_count": 0,
  "last_packet": null,
  "last_result": null,
  "last_error": null
}



{'worker_id': '666c92adca9e42a6a6214639cc5a8aa0',
 'root': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code',
 'model': 'qwen2.5-coder:7b',
 'host': 'http://localhost:11434',
 'state': 'ready',
 'created_at': '2026-04-08T01:53:21Z',
 'updated_at': '2026-04-08T01:53:21Z',
 'run_count': 0,
 'last_packet': None,
 'last_result': None,
 'last_error': None}

In [5]:
# worker run: give the model up to 90 s (8 turns × ~10 s each + acceptance test)
worker_run = run_json([
    PYTHON, "-m", "src.main",
    "worker", "run", worker_id,
    "--packet", str(task_path),
], timeout=180, check=False)

state       = worker_run.get("state", "?")
last_result = worker_run.get("last_result") or {}
stop_reason = last_result.get("stop_reason", "no result")
print(f"worker state : {state}")
print(f"stop_reason  : {stop_reason}")
print(f"tool_calls   : {last_result.get('tool_calls', [])}")


$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker run 666c92adca9e42a6a6214639cc5a8aa0 --packet /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/demo_task.json
{
  "worker_id": "666c92adca9e42a6a6214639cc5a8aa0",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "finished",
  "created_at": "2026-04-08T01:53:21Z",
  "updated_at": "2026-04-08T01:53:32Z",
  "run_count": 1,
  "last_packet": {
    "objective": "Read src/main.py lines 1 to 40 and return the list of top-level subcommands registered by build_parser().",
    "scope": "Read src/main.py only. Do not edit any file. Do not run tests.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -c \"import src.main; print('import ok')\""
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Return th

In [6]:
last_result = worker_run.get("last_result") or {}

if not last_result:
    print("No last_result recorded (worker may have errored before producing output).")
else:
    print("tool_calls   :", last_result.get("tool_calls", []))
    print("changed_files:", last_result.get("changed_files", []))
    print("stop_reason  :", last_result.get("stop_reason"))
    print()
    print("verification:")
    print(json.dumps(last_result.get("verification", {}), indent=2))
    print()
    print("final_answer:")
    print(last_result.get("final_answer") or "(empty)")


tool_calls   : ['read_file']
changed_files: []
stop_reason  : completed

verification:
{
  "acceptance_tests": [
    {
      "command": "python3 -c \"import src.main; print('import ok')\"",
      "success": true,
      "exit_code": 0,
      "output": "import ok"
    }
  ],
  "tool_runs": {
    "run_tests": [],
    "run_build": []
  }
}

final_answer:
['build_parser']


In [7]:
worker_status = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "status",
    worker_id,
])
worker_status["state"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker status 666c92adca9e42a6a6214639cc5a8aa0
{
  "worker_id": "666c92adca9e42a6a6214639cc5a8aa0",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "finished",
  "created_at": "2026-04-08T01:53:21Z",
  "updated_at": "2026-04-08T01:53:32Z",
  "run_count": 1,
  "last_packet": {
    "objective": "Read src/main.py lines 1 to 40 and return the list of top-level subcommands registered by build_parser().",
    "scope": "Read src/main.py only. Do not edit any file. Do not run tests.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -c \"import src.main; print('import ok')\""
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Return the subcommand names as a JSON array under the key 'subcommands'.",
    "escalation_policy": "Stop if

'finished'

In [8]:
worker_resumed = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "resume",
    worker_id,
    "--trace",
])
worker_resumed["run_count"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker resume 666c92adca9e42a6a6214639cc5a8aa0 --trace
{
  "worker_id": "666c92adca9e42a6a6214639cc5a8aa0",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "finished",
  "created_at": "2026-04-08T01:53:21Z",
  "updated_at": "2026-04-08T01:53:38Z",
  "run_count": 2,
  "last_packet": {
    "objective": "Read src/main.py lines 1 to 40 and return the list of top-level subcommands registered by build_parser().",
    "scope": "Read src/main.py only. Do not edit any file. Do not run tests.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -c \"import src.main; print('import ok')\""
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Return the subcommand names as a JSON array under the key 'subcommands'.",
    "escalation_policy": 

2

In [9]:
worker_closed = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "close",
    worker_id,
])
worker_closed["state"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker close 666c92adca9e42a6a6214639cc5a8aa0
{
  "worker_id": "666c92adca9e42a6a6214639cc5a8aa0",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "closed",
  "created_at": "2026-04-08T01:53:21Z",
  "updated_at": "2026-04-08T01:53:38Z",
  "run_count": 2,
  "last_packet": {
    "objective": "Read src/main.py lines 1 to 40 and return the list of top-level subcommands registered by build_parser().",
    "scope": "Read src/main.py only. Do not edit any file. Do not run tests.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -c \"import src.main; print('import ok')\""
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Return the subcommand names as a JSON array under the key 'subcommands'.",
    "escalation_policy": "Stop if Ol

'closed'

## Smoke Test Success Criteria

The smoke test is considered successful when:

- the worker is created without manual `worker_id` handling
- `worker run` returns JSON
- `last_result.tool_calls` is non-empty
- `last_result.verification.acceptance_tests` is present
- `last_result.stop_reason` is `completed` or `verification_failed`
- `worker status`, `worker resume`, and `worker close` all succeed